# 03c — Silver Layer: Fact Table — silver_ohlc_metrics

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# SILVER LAYER — FACT TABLE : silver_ohlc_metrics  (OHLC fact)
# Notebook : 03c_silver_fact_ohlc.py
#
# Data model : star schema
#   Grain     : 1 row per (coin_id, timestamp_ms) — 4h candle
#   Dim joins : dim_coin      (coin_id)
#               dim_date      (candle_date → date_id)
#               dim_date_hour (candle_datetime → date_hour_id)
#
# Output table silver_ohlc_metrics keeps the EXACT same column set
# as before — the gold layer (05_gold_trader_view) needs zero changes.
#
# What changed vs old 04_transform_ohlc:
#   • Coin metadata (segment, supply flags) joined from dim_coin.
#   • Candle timestamps joined to dim_date + dim_date_hour to expose
#     trading_session, year_month, candle_block for BI slicing.
#   • ABC validation, feature engineering, MERGE logic are unchanged.
# ══════════════════════════════════════════════════════════════════════════

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
import traceback

DB_NAME          = 'crypto_db'
BRONZE_OHLC      = f'{DB_NAME}.bronze_ohlc_data'
DIM_COIN         = f'{DB_NAME}.dim_coin'
DIM_DATE         = f'{DB_NAME}.dim_date'
DIM_DATE_HOUR    = f'{DB_NAME}.dim_date_hour'
SILVER_OHLC      = f'{DB_NAME}.silver_ohlc_metrics'
SILVER_OHLC_QUAR = f'{DB_NAME}.silver_ohlc_quarantine'
INGESTION_LOG    = f'{DB_NAME}.ingestion_status_log'

CANDLE_HOURS    = 4
CANDLES_PER_DAY = 24 // CANDLE_HOURS   # 6
LOOKBACK_DAYS   = 30

def _now_utc():
    return datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

def _today_utc():
    return datetime.now(timezone.utc).strftime('%Y-%m-%d')

print(f'Config loaded — {BRONZE_OHLC} → {SILVER_OHLC}')

In [0]:
# ── 1. Load Bronze OHLC ───────────────────────────────────────────────────

df_raw    = spark.table(BRONZE_OHLC)
total_cnt = df_raw.count()
print(f'Bronze rows : {total_cnt}')
print(f'Coins       : {df_raw.select("coin_id").distinct().count()}')
df_raw.printSchema()

In [0]:
# ── 2. Dynamic Thresholds ─────────────────────────────────────────────────
stats = df_raw.filter(F.col('open').isNotNull()&(F.col('open')>0)).select(
    F.expr('percentile_approx(abs(close-open)/open*100, 0.99)').alias('p99_body'),
    F.expr('percentile_approx(abs(close-open)/open*100, 0.90)').alias('p90_body'),
    F.expr('percentile_approx(close/open, 0.9999)').alias('p9999_ratio'),
).collect()[0]
 
ANOMALY_BODY_PCT    = float(stats['p99_body'])
MOMENTUM_SIGNAL_PCT = float(stats['p90_body'])
EXTREME_MOVE_MULT   = max(float(stats['p9999_ratio'])*3, 10.0)
print(f"Anomaly threshold: {ANOMALY_BODY_PCT:.2f}%  |  Extreme: {EXTREME_MOVE_MULT:.2f}x")
 

In [0]:
# ── 3. ABC Validation (unchanged) ────────────────────────────────────────

def apply_abc_validation(df):
    df_tagged = (df
        .withColumn('_c_nulls',
            F.when(
                F.col('open').isNull() | F.col('high').isNull() |
                F.col('low').isNull()  | F.col('close').isNull() |
                F.col('candle_datetime').isNull() | F.col('coin_id').isNull(),
                F.lit('C:null_ohlc_field')
            ).otherwise(F.lit(None)))
        .withColumn('_a_negative',
            F.when(
                (F.col('open') < 0) | (F.col('high') < 0) |
                (F.col('low')  < 0) | (F.col('close') < 0),
                F.lit('A:negative_price')
            ).otherwise(F.lit(None)))
        .withColumn('_a_zero',
            F.when(
                F.col('close').isNotNull() & (F.col('close') == 0),
                F.lit('A:zero_close')
            ).otherwise(F.lit(None)))
        .withColumn('_a_extreme',
            F.when(
                (F.col('open') > 0) &
                (F.col('close') / F.col('open') > EXTREME_MOVE_MULT),
                F.lit('A:extreme_candle')
            ).otherwise(F.lit(None)))
        .withColumn('_b_hl',
            F.when(
                F.col('high').isNotNull() & F.col('low').isNotNull() &
                (F.col('high') < F.col('low')),
                F.lit('B:high_lt_low')
            ).otherwise(F.lit(None)))
        .withColumn('_b_open',
            F.when(
                F.col('open').isNotNull() &
                ((F.col('open') < F.col('low')) | (F.col('open') > F.col('high'))),
                F.lit('B:open_outside_hl')
            ).otherwise(F.lit(None)))
        .withColumn('_b_close',
            F.when(
                F.col('close').isNotNull() &
                ((F.col('close') < F.col('low')) | (F.col('close') > F.col('high'))),
                F.lit('B:close_outside_hl')
            ).otherwise(F.lit(None)))
        .withColumn('_b_dup',
            F.when(
                F.count('*').over(Window.partitionBy('coin_id', 'timestamp_ms')) > 1,
                F.lit('B:duplicate_candle')
            ).otherwise(F.lit(None)))
    )

    reason_cols = ['_c_nulls', '_a_negative', '_a_zero', '_a_extreme',
                   '_b_hl', '_b_open', '_b_close', '_b_dup']

    df_tagged = df_tagged.withColumn(
        'quarantine_reasons',
        F.concat_ws(' | ', *[F.col(c) for c in reason_cols])
    ).drop(*reason_cols)

    df_clean = df_tagged.filter(F.col('quarantine_reasons') == '')
    df_bad   = df_tagged.filter(F.col('quarantine_reasons') != '')
    return df_clean, df_bad


df_clean, df_bad = apply_abc_validation(df_raw)
clean_cnt = df_clean.count()
bad_cnt   = df_bad.count()

print(f'Total : {total_cnt}')
print(f'Clean : {clean_cnt}  ({round(clean_cnt/total_cnt*100,1)}%)')
print(f'Bad   : {bad_cnt}  ({round(bad_cnt/total_cnt*100,1)}%)')
if bad_cnt > 0:
    df_bad.groupBy('quarantine_reasons').count().show(truncate=False)

In [0]:
# ── 4. 30-Day Window Filter ───────────────────────────────────────────────

df_clean = df_clean.filter(
    F.col('candle_datetime') >= F.date_sub(F.current_timestamp(), LOOKBACK_DAYS)
)
print(f'After 30-day filter: {df_clean.count()} rows')

In [0]:
# ── 5. Dim joins — enrich with coin master + calendar attributes ──────────

# 5a. dim_coin join
dim_coin = spark.table(DIM_COIN).select(
    F.col('id').alias('coin_id'),
    'market_segment',
    'supply_utilisation_pct',
    'ath_distance_bucket',
    'is_scarce',
)
df_clean = df_clean.join(dim_coin, on='coin_id', how='left')
print('Joined dim_coin ✅')

# 5b. dim_date join (on candle_date string → date_actual)
dim_date = spark.table(DIM_DATE).select(
    F.col('date_actual').alias('candle_date_str'),
    'date_id',
    'year_month',
    'year_quarter',
    'year_week',
    'day_name',
    'is_weekend',
)

# candle_date may be date type — cast to string for join
df_clean = df_clean.withColumn('_candle_date_str',
    F.col('candle_date').cast('string'))

df_clean = df_clean.join(
    dim_date,
    df_clean['_candle_date_str'] == dim_date['candle_date_str'],
    how='left'
).drop('candle_date_str', '_candle_date_str')
print('Joined dim_date ✅')

# 5c. dim_date_hour join (on date_id + hour extracted from candle_datetime)
dim_date_hour = spark.table(DIM_DATE_HOUR).select(
    'date_id',
    'hour',
    'trading_session',
    'candle_block',
    'is_candle_open',
)

df_clean = df_clean.withColumn('_candle_hour', F.hour('candle_datetime'))

df_clean = df_clean.join(
    dim_date_hour,
    (df_clean['date_id'] == dim_date_hour['date_id']) &
    (df_clean['_candle_hour'] == dim_date_hour['hour']),
    how='left'
).drop(dim_date_hour['date_id']).drop('hour', '_candle_hour')
print('Joined dim_date_hour ✅')

print(f'After all dim joins: {df_clean.count()} rows')

In [0]:
# ── 6. Feature Engineering (unchanged — gold depends on these) ────────────

def add_features(df):
    w = Window.partitionBy('coin_id').orderBy('candle_datetime')

    N7  = CANDLES_PER_DAY * 7  - 1
    N14 = CANDLES_PER_DAY * 14 - 1
    N30 = CANDLES_PER_DAY * 30 - 1

    w7  = w.rowsBetween(-N7,  0)
    w14 = w.rowsBetween(-N14, 0)
    w30 = w.rowsBetween(-N30, 0)

    df = (df
        .withColumn('candle_datetime',     F.col('candle_datetime').cast('timestamp'))
        .withColumn('ingestion_timestamp', F.col('ingestion_timestamp').cast('timestamp'))

        # Candle direction & body
        .withColumn('candle_direction',
            F.when(F.col('close') > F.col('open'), 'BULL')
             .when(F.col('close') < F.col('open'), 'BEAR')
             .otherwise('DOJI'))
        .withColumn('candle_body_pct',
            F.round(
                F.when(F.col('open') > 0,
                    F.abs(F.col('close') - F.col('open')) / F.col('open') * 100
                ).otherwise(F.lit(None)), 3))

        # Daily range
        .withColumn('daily_range_usd',
            F.round(F.col('high') - F.col('low'), 4))
        .withColumn('daily_range_pct',
            F.round(
                F.when(F.col('open') > 0,
                    (F.col('high') - F.col('low')) / F.col('open') * 100
                ).otherwise(F.lit(None)), 3))

        # Moving averages
        .withColumn('ma_7d',  F.round(F.avg('close').over(w7),  4))
        .withColumn('ma_14d', F.round(F.avg('close').over(w14), 4))
        .withColumn('ma_30d', F.round(F.avg('close').over(w30), 4))

        # Rolling volatility
        .withColumn('volatility_7d',  F.round(F.stddev('close').over(w7),  6))
        .withColumn('volatility_14d', F.round(F.stddev('close').over(w14), 6))
        .withColumn('volatility_30d', F.round(F.stddev('close').over(w30), 6))

        # Volatility cluster
        .withColumn('_avg_range_14d', F.avg('daily_range_usd').over(w14))
        .withColumn('volatility_cluster',
            F.when(
                F.col('daily_range_usd') > F.col('_avg_range_14d') * 1.5, 'HIGH'
            ).otherwise('NORMAL'))

        # Momentum
        .withColumn('momentum_1d',
            F.round(
                F.when(F.lag('close', 1).over(w) > 0,
                    (F.col('close') - F.lag('close', 1).over(w)) /
                     F.lag('close', 1).over(w) * 100
                ).otherwise(F.lit(None)), 2))
        .withColumn('momentum_7d',
            F.round(
                F.when(F.lag('close', N7).over(w) > 0,
                    (F.col('close') - F.lag('close', N7).over(w)) /
                     F.lag('close', N7).over(w) * 100
                ).otherwise(F.lit(None)), 2))
        .withColumn('momentum_14d',
            F.round(
                F.when(F.lag('close', N14).over(w) > 0,
                    (F.col('close') - F.lag('close', N14).over(w)) /
                     F.lag('close', N14).over(w) * 100
                ).otherwise(F.lit(None)), 2))

        # Support / resistance
        .withColumn('support_14d',    F.round(F.min('low').over(w14),  4))
        .withColumn('resistance_14d', F.round(F.max('high').over(w14), 4))
        .withColumn('support_30d',    F.round(F.min('low').over(w30),  4))
        .withColumn('resistance_30d', F.round(F.max('high').over(w30), 4))

        # Trend strength (price vs MA30)
        .withColumn('trend_strength',
            F.when(F.col('ma_30d').isNotNull() & (F.col('ma_30d') > 0),
                F.when(F.col('close') > F.col('ma_30d') * 1.05, 'STRONG_BULL')
                 .when(F.col('close') > F.col('ma_30d'),         'BULL')
                 .when(F.col('close') < F.col('ma_30d') * 0.95, 'STRONG_BEAR')
                 .when(F.col('close') < F.col('ma_30d'),         'BEAR')
                 .otherwise('NEUTRAL')
            ).otherwise('NEUTRAL'))

        # Price level signal (vs support/resistance)
        .withColumn('price_level_signal',
            F.when(F.col('close') >= F.col('resistance_14d'), 'AT_RESISTANCE')
             .when(F.col('close') <= F.col('support_14d'),    'AT_SUPPORT')
             .otherwise('MID_RANGE'))

        # Trading signal (momentum + trend confluence)
        .withColumn('trading_signal',
            F.when(
                (F.col('momentum_7d') > MOMENTUM_SIGNAL_PCT) &
                (F.col('trend_strength').isin('STRONG_BULL', 'BULL')),
                'STRONG_BUY')
             .when(
                (F.col('momentum_7d') < -MOMENTUM_SIGNAL_PCT) &
                (F.col('trend_strength').isin('STRONG_BEAR', 'BEAR')),
                'STRONG_SELL')
             .otherwise('NEUTRAL'))

        # Anomaly flag
        .withColumn('anomaly_flag',
            F.when(F.col('candle_body_pct') > ANOMALY_BODY_PCT,
                F.lit('HIGH_BODY_PCT'))
             .otherwise(F.lit(None)))

        # Data latency
        .withColumn('data_latency_hours',
            F.round(
                (F.unix_timestamp(F.current_timestamp()) -
                 F.unix_timestamp('ingestion_timestamp')) / 3600, 2))

        .drop('_avg_range_14d')
    )
    return df


df_clean = add_features(df_clean)
print(f'✅ Feature engineering done — {len(df_clean.columns)} columns')

In [0]:
# ── 7. Write silver_ohlc_metrics (idempotent MERGE) ──────────────────────
# New additive columns from dim joins:
#   date_id, year_month, year_quarter, year_week, day_name, is_weekend
#   trading_session, candle_block, is_candle_open
#   market_segment, supply_utilisation_pct, ath_distance_bucket, is_scarce
# These do NOT break any gold selects (gold uses specific column names).

df_clean.createOrReplaceTempView('_new_silver_ohlc')

if not spark.catalog.tableExists(SILVER_OHLC):
    (df_clean.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .partitionBy('coin_id', 'ingestion_date')
        .saveAsTable(SILVER_OHLC))
    print(f'Created {SILVER_OHLC}')
else:
    src_cols = set(df_clean.columns)
    tgt_cols = set(spark.table(SILVER_OHLC).columns)

    # Add new dim columns to target schema if absent
    new_cols = src_cols - tgt_cols
    for nc in new_cols:
        dtype = dict(df_clean.dtypes).get(nc, 'string')
        try:
            spark.sql(f'ALTER TABLE {SILVER_OHLC} ADD COLUMN `{nc}` {dtype}')
            print(f'Added column: {nc} ({dtype})')
        except Exception as e:
            print(f'Could not add {nc}: {e}')

    # Drop stale cols that no longer exist in source
    for sc in (tgt_cols - src_cols):
        try:
            spark.sql(f'ALTER TABLE {SILVER_OHLC} DROP COLUMN `{sc}`')
            print(f'Dropped stale column: {sc}')
        except Exception as e:
            print(f'Could not drop {sc}: {e}')

    spark.sql(f'''
        MERGE INTO {SILVER_OHLC} AS tgt
        USING _new_silver_ohlc AS src
          ON  tgt.coin_id      = src.coin_id
          AND tgt.timestamp_ms = src.timestamp_ms
        WHEN MATCHED     THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    ''')
    print(f'Merged into {SILVER_OHLC}')

silver_cnt = spark.table(SILVER_OHLC).count()
print(f'Silver rows total: {silver_cnt}')

try:
    spark.sql(f'OPTIMIZE {SILVER_OHLC} ZORDER BY (candle_datetime)')
    print('OPTIMIZE done')
except Exception as e:
    print(f'OPTIMIZE skipped: {e}')

# Quarantine
if bad_cnt > 0:
    df_q = (df_bad
        .withColumn('quarantine_timestamp', F.current_timestamp())
        .withColumn('source_table',  F.lit(BRONZE_OHLC))
        .withColumn('pipeline_stage', F.lit('ABC_VALIDATION')))
    q_exists = spark.catalog.tableExists(SILVER_OHLC_QUAR)
    (df_q.write.format('delta')
        .mode('append' if q_exists else 'overwrite')
        .option('mergeSchema', 'true')
        .partitionBy('coin_id', 'ingestion_date')
        .saveAsTable(SILVER_OHLC_QUAR))
    print(f'Quarantine updated: {bad_cnt} rows')
else:
    print('No rows quarantined')

In [0]:
# ── 8. Ingestion Log ──────────────────────────────────────────────────────

log_row = [{'log_timestamp': _now_utc(), 'ingestion_date': _today_utc(),
            'source': SILVER_OHLC, 'status': 'SUCCESS',
            'rows_written': silver_cnt, 'retry_count': 0, 'error_message': ''}]
log_sdf = spark.createDataFrame(log_row).select(
    F.col('log_timestamp').cast('timestamp'),
    'ingestion_date', 'source', 'status',
    F.col('rows_written').cast('int'),
    F.col('retry_count').cast('int'), 'error_message'
)
log_exists = spark.catalog.tableExists(INGESTION_LOG)
(log_sdf.write.format('delta')
    .mode('append' if log_exists else 'overwrite')
    .option('mergeSchema', 'true')
    .saveAsTable(INGESTION_LOG))
print('Ingestion log written')

In [0]:
# ── 9. KPI Verification (unchanged — validates gold compatibility) ─────────

def validate_kpis():
    df   = spark.table(SILVER_OHLC)
    cols = df.columns
    checks = {
        'has_ohlc':             all(c in cols for c in ['open','high','low','close']),
        'has_daily_range_pct':  'daily_range_pct'    in cols,
        'has_momentum_7d':      'momentum_7d'        in cols,
        'has_ma_30d':           'ma_30d'             in cols,
        'has_trend_strength':   'trend_strength'     in cols,
        'has_price_level':      'price_level_signal' in cols,
        'has_support_resist':   all(c in cols for c in ['support_14d','resistance_14d']),
        'has_trading_signal':   'trading_signal'     in cols,
        'has_anomaly_flag':     'anomaly_flag'       in cols,
        'has_volatility_30d':   'volatility_30d'     in cols,
        'has_data_latency':     'data_latency_hours' in cols,
        # New dim-enriched fields
        'has_date_id':          'date_id'            in cols,
        'has_trading_session':  'trading_session'    in cols,
        'has_market_segment':   'market_segment'     in cols,
        'signal_values_valid':  df.filter(
            ~F.col('trading_signal').isin('STRONG_BUY','STRONG_SELL','NEUTRAL')
        ).count() == 0,
    }
    print('\n=== VALIDATION: KPI READINESS ===')
    for k, v in checks.items():
        print(f'  {k:35} : {"PASS" if v else "FAIL"}')
    return checks


validate_kpis()
print('\n✅ silver_ohlc_metrics complete.')

In [0]:
# ── 10. Dim-enriched analytics preview ───────────────────────────────────

print('\n--- Trading session distribution ---')
spark.table(SILVER_OHLC).groupBy('trading_session').count() \
    .orderBy('count', ascending=False).show()

print('\n--- Volatility by session ---')
spark.table(SILVER_OHLC).groupBy('trading_session').agg(
    F.round(F.avg('daily_range_pct'), 3).alias('avg_range_pct'),
    F.count('*').alias('candle_count')
).orderBy('avg_range_pct', ascending=False).show()

print('\n--- Year-month candle distribution ---')
spark.table(SILVER_OHLC).groupBy('year_month').count() \
    .orderBy('year_month').show()

In [0]:
%sql
select * from crypto_db.silver_ohlc_metrics;

In [0]:
print("result")